In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import cm
from matplotlib.colors import Normalize

# ----------------------------
# Parameters you can tweak
# ----------------------------
N = 400                       # grid resolution for the unwrapped torus
texture_mode = "spin_singlet" # "spin_singlet" or "photon_polarization"
# texture_mode = "photon_polarization" # "spin_singlet" or "photon_polarization"

# Texture on the (theta_A, theta_B) torus
# We'll visualize a Born-weight-like texture R^2 that depends only on the relative angle Δ.
thetaA = np.linspace(0, 2*np.pi, N, endpoint=False)
thetaB = np.linspace(0, 2*np.pi, N, endpoint=False)
TA, TB = np.meshgrid(thetaA, thetaB, indexing="xy")
Delta = TA - TB

if texture_mode == "spin_singlet":
    # Representative relative-angle texture: cos^2(Δ/2)
    W = np.cos(Delta/2.0)**2
elif texture_mode == "photon_polarization":
    # For linear polarizers the relevant physics often appears with a double-angle: cos^2(Δ)
    W = np.cos(Delta)**2
else:
    raise ValueError("texture_mode must be 'spin_singlet' or 'photon_polarization'")

# Normalize 0..1 for consistent mapping
Wn = (W - W.min()) / (W.max() - W.min() + 1e-12)

# ----------------------------
# 1) Unwrapped torus: heatmap over [0, 2π) × [0, 2π)
# ----------------------------
fig1 = plt.figure(figsize=(8, 7))
ax1 = fig1.add_subplot(111)
im = ax1.imshow(
    Wn,
    origin="lower",
    extent=(0, 2*np.pi, 0, 2*np.pi),
    interpolation="nearest",
    aspect="equal",
)
ax1.set_title("Unwrapped torus texture on $(\\theta_A,\\theta_B) \\in [0,2\\pi)\\times[0,2\\pi)$")
ax1.set_xlabel(r"$\theta_A$")
ax1.set_ylabel(r"$\theta_B$")
cbar = fig1.colorbar(im, ax=ax1, fraction=0.046, pad=0.04)
cbar.set_label(r"Normalized weight $W(\theta_A,\theta_B)$")

ticks = [0, np.pi/2, np.pi, 3*np.pi/2, 2*np.pi]
ticklabels = ["0", r"$\pi/2$", r"$\pi$", r"$3\pi/2$", r"$2\pi$"]
ax1.set_xticks(ticks, ticklabels)
ax1.set_yticks(ticks, ticklabels)

plt.tight_layout()
plt.show()

# ----------------------------
# 2) 3D torus with the same texture mapped onto its surface
# ----------------------------
# Parametrize a geometric torus in 3D:
# u = poloidal angle, v = toroidal angle
# Here we'll map theta_A -> u and theta_B -> v to give the same joint-angle coordinates.
Nu, Nv = 160, 240
u = np.linspace(0, 2*np.pi, Nu, endpoint=False)
v = np.linspace(0, 2*np.pi, Nv, endpoint=False)
U, V = np.meshgrid(u, v, indexing="xy")

# Torus geometry parameters
R = 2.5   # major radius
r = 1.0   # minor radius
X = (R + r*np.cos(U)) * np.cos(V)
Y = (R + r*np.cos(U)) * np.sin(V)
Z = r*np.sin(U)

# Compute the same texture on the (U,V) grid
Delta_uv = U - V
if texture_mode == "spin_singlet":
    W_uv = np.cos(Delta_uv/2.0)**2
else:
    W_uv = np.cos(Delta_uv)**2
W_uvn = (W_uv - W_uv.min()) / (W_uv.max() - W_uv.min() + 1e-12)

# Use the *default* colormap (no explicit color choice) to map weight -> facecolors
cmap = plt.get_cmap(None)
norm = Normalize(vmin=0.0, vmax=1.0)
facecolors = cmap(norm(W_uvn))

fig2 = plt.figure(figsize=(9, 7))
ax2 = fig2.add_subplot(111, projection="3d")
ax2.plot_surface(
    X, Y, Z,
    rstride=1, cstride=1,
    facecolors=facecolors,
    linewidth=0,
    antialiased=False,
    shade=False,
)
ax2.set_title("3D torus with joint-angle texture mapped onto its surface")
ax2.set_axis_off()

# Add a colorbar (linked to the same normalization)
mappable = cm.ScalarMappable(norm=norm, cmap=cmap)
mappable.set_array(W_uvn)
cbar2 = fig2.colorbar(mappable, ax=ax2, fraction=0.046, pad=0.04)
cbar2.set_label(r"Normalized weight $W(\theta_A,\theta_B)$")

# A nice default view angle
ax2.view_init(elev=25, azim=35)

plt.tight_layout()
plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# ----------------------------
# Choose analyzer settings (example Δ = 45°)
# ----------------------------
a = 0.0
b = np.pi/4  # 45 degrees
Delta_ab = (b - a)

# ----------------------------
# Texture definition on (θA, θB)
# ----------------------------
N = 500
thetaA = np.linspace(0, 2*np.pi, N, endpoint=False)
thetaB = np.linspace(0, 2*np.pi, N, endpoint=False)
TA, TB = np.meshgrid(thetaA, thetaB, indexing="xy")

# Use a representative entangled-like Born weight on the joint-angle torus
# (spin-singlet-flavored relative-angle texture)
W = np.cos((TA - TB)/2.0)**2

# ----------------------------
# Re-express axes so the ± bins are simple halves:
# Define + region for each side as θ in [setting-π/2, setting+π/2)
# Then in shifted coordinates x,y, the boundary is just x=π and y=π.
# This is just a coordinate shift (a torus "cut" choice).
# ----------------------------
shiftA = (a - np.pi/2) % (2*np.pi)
shiftB = (b - np.pi/2) % (2*np.pi)

kA = int(round(shiftA / (2*np.pi) * N))  # columns (θA axis)
kB = int(round(shiftB / (2*np.pi) * N))  # rows (θB axis)

W2 = np.roll(W, -kA, axis=1)  # shift θA so that θA=shiftA is at x=0
W2 = np.roll(W2, -kB, axis=0) # shift θB so that θB=shiftB is at y=0

# Normalize for display only
W2n = (W2 - W2.min()) / (W2.max() - W2.min() + 1e-12)

# ----------------------------
# Quadrant weights under these bins
# (+) means x in [0,π), (-) means x in [π,2π)
# (+) means y in [0,π), (-) means y in [π,2π)
# ----------------------------
half = N // 2
Wtot = W2.sum()

P_pp = W2[:half, :half].sum() / Wtot
P_pm = W2[:half, half:].sum() / Wtot
P_mp = W2[half:, :half].sum() / Wtot
P_mm = W2[half:, half:].sum() / Wtot

# Correlation under ± labeling
E = (P_pp + P_mm) - (P_pm + P_mp)

# Differences vs uniform 0.25 (helps "gain/lose")
def d25(p): 
    return p - 0.25

# ----------------------------
# Plot: unwrapped torus with bin boundaries and quadrant totals
# ----------------------------
fig = plt.figure(figsize=(9, 8))
ax = fig.add_subplot(111)

im = ax.imshow(
    W2n,
    origin="lower",
    extent=(0, 2*np.pi, 0, 2*np.pi),
    interpolation="nearest",
    aspect="equal",
)

# Bin boundaries (single vertical + single horizontal line)
ax.axvline(np.pi, linewidth=2)
ax.axhline(np.pi, linewidth=2)

ax.set_title(
    "Unwrapped torus with outcome bins\n"
    f"Settings: a = {a*180/np.pi:.1f}°, b = {b*180/np.pi:.1f}°  (Δ = {Delta_ab*180/np.pi:.1f}°)"
)
ax.set_xlabel("Alice coordinate x (shifted so + is left half)")
ax.set_ylabel("Bob coordinate y (shifted so + is bottom half)")

ticks = [0, np.pi/2, np.pi, 3*np.pi/2, 2*np.pi]
ticklabels = ["0", r"$\pi/2$", r"$\pi$", r"$3\pi/2$", r"$2\pi$"]
ax.set_xticks(ticks, ticklabels)
ax.set_yticks(ticks, ticklabels)

cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
cbar.set_label("Normalized weight W(x,y)")

# Annotate quadrant stats
# Quadrant centers
centers = {
    "++": (np.pi/2, np.pi/2),
    "+-": (3*np.pi/2, np.pi/2),
    "-+": (np.pi/2, 3*np.pi/2),
    "--": (3*np.pi/2, 3*np.pi/2),
}
vals = {
    "++": P_pp,
    "+-": P_pm,
    "-+": P_mp,
    "--": P_mm,
}
for key, (cx, cy) in centers.items():
    p = vals[key]
    delta = d25(p)
    ax.text(
        cx, cy,
        f"{key}\nP={p:.3f}\nΔP={delta:+.3f}",
        ha="center", va="center",
        fontsize=12,
        bbox=dict(boxstyle="round", alpha=0.75),
    )

# Global correlation annotation
ax.text(
    0.02, 0.98,
    f"E = (P++ + P--) - (P+- + P-+) = {E:.3f}",
    transform=ax.transAxes,
    ha="left", va="top",
    fontsize=12,
    bbox=dict(boxstyle="round", alpha=0.75),
)

plt.tight_layout()
plt.show()
